In [ ]:
from ultralytics import YOLO

In [ ]:
model = YOLO("../best_models/best.pt")

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

# Path to test folder
test_folder = "test/images"

# Get all image files
image_files = [f for f in os.listdir(test_folder) if f.lower().endswith((".png", ".jpg", ".jpeg"))]

print(f"Found {len(image_files)} images in test folder\n")

# Process each image
for image_file in image_files:
    image_path = os.path.join(test_folder, image_file)
    
    # Load image
    img = Image.open(image_path).convert("RGB")
    
    # Run prediction once
    print(f"\nPredicting for: {image_file}")
    results = model(image_path)
    annotated_img = results[0].plot()
    
    # Side-by-side display: original vs annotated
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    axes[0].imshow(img)
    axes[0].axis("off")
    axes[0].set_title(f"Original: {image_file}", fontsize=12)
    
    axes[1].imshow(annotated_img)
    axes[1].axis("off")
    axes[1].set_title("Predictions", fontsize=12)
    
    plt.tight_layout()
    plt.show()
    
    # Print detections
    for result in results:
        boxes = result.boxes
        if len(boxes) == 0:
            print("  No detections.")
        for box in boxes:
            class_id = int(box.cls[0])
            confidence = float(box.conf[0])
            class_name = result.names[class_id]
            print(f"  Detected: {class_name} (Confidence: {confidence:.2%})")
    
    print("-" * 50)

In [ ]:
import cv2
from IPython.display import display, clear_output
import numpy as np

# Path to test folder
test_folder = "test/videos"

# Get all video files
video_files = [f for f in os.listdir(test_folder) if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))]

print(f"Found {len(video_files)} videos in test folder\n")

# Process each video
for video_file in video_files:
    video_path = os.path.join(test_folder, video_file)
    
    print(f"\nProcessing video: {video_file}")
    print("-" * 50)
    
    # Open video
    cap = cv2.VideoCapture(video_path)
    
    # Get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"FPS: {fps}, Total Frames: {total_frames}")
    
    frame_count = 0
    display_every = max(1, fps // 2)  # Display every half second
    
    while cap.isOpened():
        ret, frame = cap.read()
        
        if not ret:
            break
        
        frame_count += 1
        
        # Process every Nth frame to avoid overwhelming output
        if frame_count % display_every == 0:
            # Convert BGR to RGB
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            
            # Make prediction
            results = model(frame)
            
            # Get annotated frame
            annotated_frame = results[0].plot()
            
            # Display side by side
            fig, axes = plt.subplots(1, 2, figsize=(16, 6))
            
            # Original frame
            axes[0].imshow(frame_rgb)
            axes[0].axis('off')
            axes[0].set_title(f'Original Frame {frame_count}/{total_frames}', fontsize=12)
            
            # Annotated frame
            axes[1].imshow(annotated_frame)
            axes[1].axis('off')
            axes[1].set_title(f'Predictions', fontsize=12)
            
            plt.tight_layout()
            plt.show()
            
            # Print detections
            for result in results:
                boxes = result.boxes
                if len(boxes) > 0:
                    print(f"Frame {frame_count} detections:")
                    for box in boxes:
                        class_id = int(box.cls[0])
                        confidence = float(box.conf[0])
                        class_name = result.names[class_id]
                        print(f"  {class_name} ({confidence:.2%})")
    
    cap.release()
    print(f"\nFinished processing: {video_file}")
    print("=" * 50)